# [Do Deep Nets Really Need to be Deep?](https://arxiv.org/pdf/1312.6184)

## Introduction

A shallow neural network is capable of <ins>representing</ins> a more accurate function of the data through **Model Compression**, when compared to training the model on the original data set.

Therefore, the complexity of the underlying function and size of representation best used to <ins>learn</ins> the function are different.

**Gap**: Previous research assumed depth gives representational advantage, but wasn't questioned whether it was required in practice.

**Improvement**: Shows depth advantage is not about representational limits since a shallow net can approximate same function learned by deep net (**also sets foundation for original distillation paper**)

## Approach 

**Soft Targets**

It's better to train a student model on logits since different logits can map to same distribution when using softmax (technically losing information the complex model learned)

Also, softmax can lead to few large values relative to others, which would cause cross entropy to focus on them, ignoring others <br><br>

In [1]:
import torch
print(torch.nn.functional.softmax(torch.tensor([-10.0, 0.0, 10.0]), dim=-1))
print(torch.nn.functional.softmax(torch.tensor([10.0, 20.0, 30.0]), dim=-1))

tensor([2.0611e-09, 4.5398e-05, 9.9995e-01])
tensor([2.0611e-09, 4.5398e-05, 9.9995e-01])


Say softmax target is $[2.06e^{-9}, 4.53e^{-5}, 9.99e^{-1}]$ and prediction is $[\frac{1}{3}, \frac{1}{3}, \frac{1}{3}]$

$CE_{Loss} \approx -[3.0e^{-7} \ log(\frac{1}{3}) + 6.7e^{-3} \ log(\frac{1}{3}) + 9.9e^{-1} \ log(\frac{1}{3})]$

We can see most of the loss would come from largest dimension of the target, so the student model would focus mostly on matching that value.


Now consider a logits target of $[-10.0, 0.0, 10.0]$ and prediction is $[5, 5, 5]$

$MSE_{Loss} \approx \frac{1}{3}  [(-10.0 - 5)^2 + (0.0 - 5)^2 + (10.0 - 5)^2]  = \frac{1}{3}  [(-15)^2 + (5)^2 + (5)^2]$

Each dimension of target and prediction contribute substantially to the loss. 

## Result

**Why does Model Compression perform better than training on original dataset? (Even without using extra data)**

Can be due to: 
* Label noise: Teacher model can eliminate label errors by predicting those correctly (from generalization)
* Function complexity reduction: Teacher model simplifies region of true distribution that are hard to learn
* Uncertainty information: Teacher model soft targets provide more information than hard targets, e.g. confusable classes

**As accuracy of teacher model increases linearly, what happens to student model accuracy?**

The paper shows its nearly linear which means: 
* Improvements in the teacher lead to proportional improvements in the student
* The more accurate the teacher, the more efficient you can go in the smaller model
* The smaller models tried in the paper don't run out of capacity 

## Application

### Data

In [2]:
import torch
from torch import nn
import torch.nn.functional as F

# Large (complex) model with regularization (dropout)
class LargeNet(nn.Module):
    def __init__(self, input_size=(28,28), output_size=10, num_neurons=800, p=0.20):
        super(LargeNet, self).__init__()
        input_size = np.prod(input_size)
        self.fc1 = nn.Linear(input_size, num_neurons)
        self.fc2 = nn.Linear(num_neurons, num_neurons)
        self.out = nn.Linear(num_neurons, output_size)
        self.dropout = nn.Dropout(p)

    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        
        x = self.fc2(x)
        x = F.relu(x)
        x = self.dropout(x)
        
        logits = self.out(x)
        return logits

# Small (distilled) model without regularization
class SmallNet(nn.Module):
    def __init__(self, input_size=(28,28), output_size=10, num_neurons=200):
        super(SmallNet, self).__init__()
        input_size = np.prod(input_size)
        self.fc1 = nn.Linear(input_size, num_neurons)
        self.fc2 = nn.Linear(num_neurons, num_neurons)
        self.out = nn.Linear(num_neurons, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        
        x = self.fc2(x)
        x = F.relu(x)
        
        logits = self.out(x)
        return logits

In [3]:
# Data
from torchvision import datasets, transforms

# Jitter 2 pixels
jitter = 2 / 28
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomAffine(degrees=0, translate=(jitter, jitter)),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
])

# Loading MNIST data
train_data = datasets.MNIST(root='data', train=True, download=True, transform=train_transform)
test_data = datasets.MNIST(root='data', train=False, download=True, transform=test_transform)

# Create data loaders
BATCH_SIZE = 128
train_loader = torch.utils.data.DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               RandomAffine(degrees=[0.0, 0.0], translate=(0.07142857142857142, 0.07142857142857142))
           )

In [4]:
from tqdm.notebook import tqdm
import copy

def train_model(model, optimizer, num_epochs=20, gamma=0.95):
    model = model.to(device)
    
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)

    best_model_state = None
    best_model_errors = float("inf")
    for epoch in range(num_epochs):
        model.train()  # Set the model to training mode
        running_loss = 0.0
    
        progress_bar = tqdm(total=len(train_loader))
    
        for inputs, labels in train_loader:
            inputs, labels = inputs.view(inputs.shape[0], -1).to(device), labels.to(device)
            optimizer.zero_grad()  # Zero the gradients
            
            outputs = model(inputs)  # Forward pass
            loss = criterion(outputs, labels)  # Compute the loss
            loss.backward()  # Backward pass
            optimizer.step()  # Update the weights
            
            running_loss += loss.item() * inputs.size(0)
            
            progress_bar.update(1)
            progress_bar.set_description(f"Loss: {loss.item():.4f}")
        scheduler.step()

        next_lr = optimizer.param_groups[0]["lr"]
        all_preds, all_labels = test_model(model)
        num_instances = len(all_preds)
        correct = (all_preds == all_labels).sum()
        accuracy =  correct / num_instances
        errors = num_instances - correct
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Accuracy: {accuracy}, Errors: {errors}, next LR: {next_lr}")

        # Save best model
        if errors < best_model_errors:
            best_model_errors = errors
            best_model_state = copy.deepcopy(model.state_dict())
            print(f"Saved new best model with {best_model_errors} test errors")

    # Load best model back
    model.load_state_dict(best_model_state)
    print(f"Loaded best model with {best_model_errors} test errors")
    return model

def test_model(model):
    model.eval()
    
    all_preds = []
    all_labels = []

    progress_bar = tqdm(total=len(test_loader))

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.view(inputs.shape[0], -1).to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            preds = outputs.argmax(dim=-1)

            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

            progress_bar.update(1)

    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    return all_preds, all_labels

def load_model(model, path, device):
    state_dict = torch.load(path)
    model.load_state_dict(state_dict)
    return model.to(device)

In [5]:
import numpy as np 

LR = 1e-1
MOMENTUM = 0.9
large_model = LargeNet()
optimizer = torch.optim.SGD(large_model.parameters(), lr=LR, momentum=MOMENTUM)
criterion = nn.CrossEntropyLoss()
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [17]:
# large_model = train_model(large_model, optimizer, num_epochs=50)

large_model = load_model(large_model, path='ckpts/large_model.pt', device=device)
all_preds, all_labels = test_model(large_model)

accuracy = (all_preds == all_labels).float().mean().item()
errors = (all_preds != all_labels).sum().item()
print(f"Accuracy: {accuracy:.4f}, Errors: {errors}")

C:\Users\angel\AppData\Local\Temp\ipykernel_3256\2572169861.py:79: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path)


  0%|          | 0/79 [00:00<?, ?it/s]

Accuracy: 0.9935, Errors: 65


In [31]:
def model_compression(small_model, large_model, optimizer, num_epochs=15, gamma=0.95):
    small_model = small_model.to(device)
    large_model = large_model.to(device)
    large_model.eval()

    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)
    
    for epoch in range(num_epochs):
        small_model.train()  # Set the small_model to training mode
        running_loss = 0.0
    
        progress_bar = tqdm(total=len(train_loader))
    
        for inputs, labels in train_loader:
            inputs, labels = inputs.view(inputs.shape[0], -1).to(device), labels.to(device)
            optimizer.zero_grad()  # Zero the gradients

            # Forward passes
            outputs_small = small_model(inputs)
            outputs_large = large_model(inputs)
            
            with torch.no_grad():
                teacher_probs = F.softmax(outputs_large, dim=1)
            student_log_probs = F.log_softmax(outputs_small, dim=1)

            # Compute the loss
            loss = F.kl_div(student_log_probs, teacher_probs, reduction="batchmean")
            
            loss.backward()  # Backward pass
            optimizer.step()  # Update the weights
            
            running_loss += loss.item() * inputs.size(0)
            
            progress_bar.update(1)
            progress_bar.set_description(f"Loss: {loss.item():.4f}")
        scheduler.step()

        next_lr = optimizer.param_groups[0]["lr"]
        
        all_preds, all_labels = test_model(small_model)
        num_instances = len(all_preds)
        correct = (all_preds == all_labels).sum()
        accuracy =  correct / num_instances
        errors = num_instances - correct
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Accuracy: {accuracy}, Errors: {errors}, next LR: {next_lr}")

In [32]:
LR = 1e-1
MOMENTUM = 0.9
small_model = SmallNet()
optimizer = torch.optim.SGD(small_model.parameters(), lr=LR, momentum=MOMENTUM)
criterion = nn.CrossEntropyLoss()

In [33]:
model_compression(small_model, large_model, optimizer, num_epochs=30)

  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/79 [00:00<?, ?it/s]

Epoch 1/30, Loss: 0.4484, Accuracy: 0.9632999897003174, Errors: 367, next LR: 0.095


  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/79 [00:00<?, ?it/s]

...

  0%|          | 0/79 [00:00<?, ?it/s]

Epoch 29/30, Loss: 0.0108, Accuracy: 0.9908000230789185, Errors: 92, next LR: 0.022593554099256556


  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/79 [00:00<?, ?it/s]

Epoch 30/30, Loss: 0.0107, Accuracy: 0.9907000064849854, Errors: 93, next LR: 0.021463876394293726


In [35]:
def model_compression_logits(small_model, large_model, optimizer, num_epochs=15, gamma=0.95):
    small_model = small_model.to(device)
    large_model = large_model.to(device)
    large_model.eval()

    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)
    
    for epoch in range(num_epochs):
        small_model.train()  # Set the small_model to training mode
        running_loss = 0.0
    
        progress_bar = tqdm(total=len(train_loader))
    
        for inputs, labels in train_loader:
            inputs, labels = inputs.view(inputs.shape[0], -1).to(device), labels.to(device)
            optimizer.zero_grad()  # Zero the gradients

            # Forward passes
            outputs_small = small_model(inputs)
            outputs_large = large_model(inputs)

            # Compute the loss
            t_logits = outputs_large - outputs_large.mean(dim=1, keepdim=True)
            s_logits = outputs_small - outputs_small.mean(dim=1, keepdim=True)
            loss = F.mse_loss(s_logits, t_logits)
            
            loss.backward()  # Backward pass
            optimizer.step()  # Update the weights
            
            running_loss += loss.item() * inputs.size(0)
            
            progress_bar.update(1)
            progress_bar.set_description(f"Loss: {loss.item():.4f}")
        scheduler.step()

        next_lr = optimizer.param_groups[0]["lr"]
        
        all_preds, all_labels = test_model(small_model)
        num_instances = len(all_preds)
        correct = (all_preds == all_labels).sum()
        accuracy =  correct / num_instances
        errors = num_instances - correct
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Accuracy: {accuracy}, Errors: {errors}, next LR: {next_lr}")

In [36]:
LR = 1e-2
MOMENTUM = 0.9
small_model = SmallNet()
optimizer = torch.optim.SGD(small_model.parameters(), lr=LR, momentum=MOMENTUM)
criterion = nn.CrossEntropyLoss()

In [37]:
model_compression_logits(small_model, large_model, optimizer, num_epochs=30)

  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/79 [00:00<?, ?it/s]

Epoch 1/30, Loss: 11.7033, Accuracy: 0.9757000207901001, Errors: 243, next LR: 0.0095


  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/79 [00:00<?, ?it/s]

...

  0%|          | 0/79 [00:00<?, ?it/s]

Epoch 29/30, Loss: 0.6315, Accuracy: 0.9911999702453613, Errors: 88, next LR: 0.0022593554099256553


  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/79 [00:00<?, ?it/s]

Epoch 30/30, Loss: 0.6304, Accuracy: 0.9909999966621399, Errors: 90, next LR: 0.0021463876394293723
